# V3 — Subject-Adaptive FBGAN + Spatial-Temporal EEG Transformer
### PhysioNet EEG Motor Movement/Imagery Dataset

This is the research-oriented successor to the previous CRNN-DF implementation.

It keeps the central ideas of Zhang et al. (Frontiers in Neuroscience, 2023):

**filter bank → OVR-CSP → LASSO sparse spatial features → FBGAN target adaptation → discriminative feature learning**

and adds a compact EEG Transformer inspired by convolutional-Transformer EEG work.

### Key improvements

1. The previous experiment used only four source subjects per LOSO fold (`360` training trials vs. `90` target trials). This version uses a larger source pool.
2. The FBGAN stage is actually trained and used for target-specific augmentation.
3. FBCSP/LASSO is fitted on target calibration data for the held-out subject, while final target test trials remain untouched.
4. The classifier is a spatial-temporal CNN + Transformer.
5. Source-only pretraining is followed by source + target-like synthetic adaptive fine-tuning.
6. Optional CORAL feature alignment is added during adaptive fine-tuning.
7. Subject quality ranking identifies a high-quality development cohort without selecting subjects using classifier test accuracy.
8. Important tensor dimensions can be traced with `debug=True`.

> Zhang et al. report 72.74 ± 10.44% on BCI Competition IV-2a. This notebook treats that number as a methodological reference, not as a guaranteed PhysioNet target.

In [1]:
# ============================================================
# CELL 1 - Environment, Imports, Reproducibility
# ============================================================

import os
import json
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import butter, sosfiltfilt
from scipy.linalg import eigh

from sklearn.linear_model import Lasso
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

import mne

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

SEED = 42


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


@dataclass
class Config:
    # Dataset
    dataset_root: str = (
        "/Users/ashokvarmabevara/MtechProj/eegmmidb"
    )
    sfreq: int = 160
    channels: int = 64
    seconds: float = 4.0
    samples: int = 640
    n_classes: int = 4

    # Subject selection
    subject_mode: str = "best_development"
    quality_scan_all_109: bool = True
    n_source_subjects: int = 20
    n_target_subjects: int = 5
    source_validation_subject_fraction: float = 0.10
    manual_source_subjects: tuple = ()
    manual_target_subjects: tuple = ()

    # Target adaptation
    target_calibration_per_class: int = 10

    # Preprocessing
    filter_order: int = 5
    low_hz: float = 1.0
    high_hz: float = 38.0
    bands: tuple = (
        (1, 4), (4, 8), (8, 12), (12, 16), (16, 20),
        (20, 24), (24, 28), (28, 32), (32, 35), (35, 38),
    )

    # FBCSP + LASSO
    csp_vectors_per_ovr_class: int = 4
    lasso_alphas: tuple = (
        1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1,
    )
    lasso_cv: int = 5
    max_sparse_features: int = 64

    # FBGAN
    latent_dim: int = 256
    gan_batch_size: int = 5
    gan_lr: float = 1e-4
    gan_epochs: int = 30
    fake_per_class: int = 750
    gan_clip: float = 3.0
    run_fbgan: bool = True

    # Transformer classifier
    embed_dim: int = 64
    n_heads: int = 4
    n_transformer_layers: int = 3
    ff_dim: int = 128
    transformer_dropout: float = 0.30
    classifier_lr: float = 1e-4
    fine_tune_lr: float = 5e-5
    weight_decay: float = 1e-4
    batch_size: int = 32
    pretrain_epochs: int = 40
    finetune_epochs: int = 60
    early_stopping_patience: int = 12

    # Discriminative feature loss
    lambda_center: float = 0.10
    center_alpha: float = 0.02
    center_update_every: int = 15

    # Domain alignment
    use_coral: bool = True
    lambda_coral: float = 0.03

    # Runtime
    num_workers: int = 0
    debug_shapes: bool = False

    # Output
    result_root: str = (
        "./results_v3_hybrid_transformer"
    )


CFG = Config()

DATASET_ROOT = Path(CFG.dataset_root)
RESULT_ROOT = Path(CFG.result_root)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Dataset:", DATASET_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())
print("Results:", RESULT_ROOT.resolve())

Device: mps
Dataset: /Users/ashokvarmabevara/MtechProj/eegmmidb
Dataset exists: True
Results: /Users/ashokvarmabevara/MtechProj/results_v3_hybrid_transformer


## Cell 2 — Architecture overview

```text
                    PHYSIONET EEG
                   (B, 1, 64, 640)
                          │
            ┌─────────────┴─────────────┐
            │                           │
        FILTER BANK                 Raw EEG
     10 bands, 1–38 Hz                 │
            │                          │
        OVR-CSP                       CNN
            │                    spatial-temporal
         LASSO                        encoder
            │                           │
    sparse spatial filters       40 temporal tokens
            │                           │
            └───────► FBGAN ◄──────────┘
                     │   │
                    Dφ   Dψ
                     │   │
                     └───┘
                       │
          target-specific synthetic EEG
                       │
            source EEG + synthetic EEG
                       │
             spatial-temporal CNN
                       │
              Transformer Encoder
                       │
                 64-D feature
                       │
       ┌───────────────┴───────────────┐
       │                               │
 Cross Entropy                 Center Distance Loss
       │                               │
       └───────────────┬───────────────┘
                       │
                    CORAL
               (optional domain
                 alignment)
                       │
                       ▼
                 4-class MI
```

The Transformer is inserted after an EEG-specific spatial-temporal convolutional front-end rather than feeding all 640 samples as tokens. This follows the general local-feature + global-attention design used by EEG convolutional-Transformer models.

In [2]:
# ============================================================
# CELL 3 - Local PhysioNet EEGMMIDB loader
# ============================================================

LEFT_RIGHT_RUNS = [4, 8, 12]
HANDS_FEET_RUNS = [6, 10, 14]
ALL_MI_RUNS = LEFT_RIGHT_RUNS + HANDS_FEET_RUNS

CLASS_NAMES = {
    0: "Left Fist",
    1: "Right Fist",
    2: "Both Fists",
    3: "Both Feet",
}

EXPECTED_CHANNEL_NAMES = None


def check_dataset():
    if not DATASET_ROOT.exists():
        raise FileNotFoundError(
            f"Dataset not found: {DATASET_ROOT}"
        )

    subject_dirs = sorted(
        [
            p for p in DATASET_ROOT.iterdir()
            if p.is_dir() and p.name.startswith("S")
        ]
    )

    subject_ids = []
    for path in subject_dirs:
        suffix = path.name[1:]
        if suffix.isdigit():
            subject_ids.append(int(suffix))

    subject_ids = sorted(subject_ids)

    print("=" * 72)
    print("PHYSIONET EEGMMIDB")
    print("=" * 72)
    print("Root:", DATASET_ROOT)
    print("Subjects found:", len(subject_ids))
    print("First subjects:", subject_ids[:10])
    print("Last subjects:", subject_ids[-10:])
    print("=" * 72)

    return subject_ids


AVAILABLE_SUBJECTS = check_dataset()


def resolve_edf(subject_id, run_id):
    subject_dir = DATASET_ROOT / f"S{subject_id:03d}"

    exact = (
        subject_dir
        / f"S{subject_id:03d}R{run_id:02d}.edf"
    )

    if exact.exists():
        return exact

    candidates = list(
        subject_dir.glob(f"*R{run_id:02d}.edf")
    )

    return candidates[0] if candidates else None


def prepare_raw(raw):
    global EXPECTED_CHANNEL_NAMES

    try:
        raw.pick("eeg")
    except Exception:
        raw.pick_types(eeg=True)

    try:
        raw = mne.datasets.eegbci.standardize(
            raw,
            verbose=False,
        )
    except Exception:
        pass

    if not np.isclose(
        float(raw.info["sfreq"]),
        CFG.sfreq,
    ):
        raw.resample(
            CFG.sfreq,
            npad="auto",
            verbose=False,
        )

    if EXPECTED_CHANNEL_NAMES is None:
        if len(raw.ch_names) < CFG.channels:
            raise RuntimeError(
                f"Only {len(raw.ch_names)} EEG channels found."
            )
        EXPECTED_CHANNEL_NAMES = raw.ch_names[:CFG.channels]

    missing = [
        ch for ch in EXPECTED_CHANNEL_NAMES
        if ch not in raw.ch_names
    ]

    if missing:
        raise RuntimeError(
            f"Missing channels: {missing[:10]}"
        )

    raw.pick(EXPECTED_CHANNEL_NAMES)

    return raw


def run_label_mapping(run_id):
    if run_id in LEFT_RIGHT_RUNS:
        return {"T1": 0, "T2": 1}

    if run_id in HANDS_FEET_RUNS:
        return {"T1": 2, "T2": 3}

    raise ValueError(f"Unsupported MI run: {run_id}")


def load_run_trials(subject_id, run_id):
    path = resolve_edf(subject_id, run_id)

    if path is None:
        return None, None

    try:
        raw = mne.io.read_raw_edf(
            path,
            preload=True,
            verbose=False,
        )
        raw = prepare_raw(raw)

        events, event_id = (
            mne.events_from_annotations(
                raw,
                verbose=False,
            )
        )

        inverse_event_id = {
            code: name
            for name, code in event_id.items()
        }

        mapping = run_label_mapping(run_id)

        xs = []
        ys = []

        for event in events:
            description = inverse_event_id.get(
                int(event[2]),
                None,
            )

            if description not in mapping:
                continue

            start = int(event[0])
            stop = start + CFG.samples

            if start < 0 or stop > raw.n_times:
                continue

            trial = raw.get_data(
                start=start,
                stop=stop,
            )

            if trial.shape != (
                CFG.channels,
                CFG.samples,
            ):
                continue

            trial = np.asarray(
                trial,
                dtype=np.float32,
            )

            trial = np.nan_to_num(
                trial,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            xs.append(trial)
            ys.append(mapping[description])

        if not xs:
            return None, None

        return (
            np.stack(xs).astype(np.float32),
            np.asarray(ys, dtype=np.int64),
        )

    except Exception as exc:
        print(
            f"[WARNING] S{subject_id:03d} "
            f"R{run_id:02d}: {exc}"
        )
        return None, None


def load_subject(subject_id):
    xs = []
    ys = []

    for run_id in ALL_MI_RUNS:
        x_run, y_run = load_run_trials(
            subject_id,
            run_id,
        )

        if x_run is None:
            continue

        xs.append(x_run)
        ys.append(y_run)

    if not xs:
        return None, None

    x = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)

    assert x.shape[1:] == (
        CFG.channels,
        CFG.samples,
    )

    return x, y


def split_subject_calibration(
    x,
    y,
    calibration_per_class,
    seed=SEED,
):
    rng = np.random.default_rng(seed)

    calibration_idx = []
    test_idx = []

    for class_id in range(CFG.n_classes):
        idx = np.flatnonzero(y == class_id)

        if len(idx) <= calibration_per_class:
            raise ValueError(
                f"Class {class_id} has only {len(idx)} "
                f"trials; cannot reserve "
                f"{calibration_per_class}."
            )

        shuffled = rng.permutation(idx)

        calibration_idx.extend(
            shuffled[:calibration_per_class]
        )
        test_idx.extend(
            shuffled[calibration_per_class:]
        )

    calibration_idx = np.asarray(
        calibration_idx,
        dtype=int,
    )
    test_idx = np.asarray(
        test_idx,
        dtype=int,
    )

    rng.shuffle(calibration_idx)
    rng.shuffle(test_idx)

    return (
        x[calibration_idx],
        y[calibration_idx],
        x[test_idx],
        y[test_idx],
    )


def cache_subject(subject_id):
    cache_dir = RESULT_ROOT / "subject_cache"
    cache_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    x_file = cache_dir / f"S{subject_id:03d}_X.npy"
    y_file = cache_dir / f"S{subject_id:03d}_y.npy"

    if x_file.exists() and y_file.exists():
        return (
            np.load(x_file),
            np.load(y_file),
        )

    x, y = load_subject(subject_id)

    if x is None:
        return None, None

    np.save(x_file, x)
    np.save(y_file, y)

    return x, y


test_x, test_y = cache_subject(
    AVAILABLE_SUBJECTS[0]
)

print("\nSmoke test:")
print("Subject:", AVAILABLE_SUBJECTS[0])
print("X:", None if test_x is None else test_x.shape)
print("y:", None if test_y is None else test_y.shape)

PHYSIONET EEGMMIDB
Root: /Users/ashokvarmabevara/MtechProj/eegmmidb
Subjects found: 109
First subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Last subjects: [100, 101, 102, 103, 104, 105, 106, 107, 108, 109]

Smoke test:
Subject: 1
X: (90, 64, 640)
y: (90,)


In [3]:
# ============================================================
# CELL 4 - Subject quality ranking + experiment cohort
# ============================================================

def subject_quality_row(subject_id):
    x, y = cache_subject(subject_id)

    if x is None:
        return {
            "subject": subject_id,
            "valid": 0,
            "trials": 0,
            "classes_present": 0,
            "class_balance": 0.0,
            "flat_ratio": 1.0,
            "artifact_ratio": 1.0,
            "median_abs": np.inf,
            "quality_score": -np.inf,
        }

    class_counts = np.bincount(
        y,
        minlength=CFG.n_classes,
    )

    class_balance = (
        class_counts.min()
        / max(class_counts.max(), 1)
    )

    per_trial_std = np.std(
        x,
        axis=(1, 2),
    )

    flat_ratio = np.mean(
        per_trial_std < 1e-8
    )

    trial_median = np.median(
        x,
        axis=(1, 2),
        keepdims=True,
    )

    mad = np.median(
        np.abs(
            x - trial_median
        ),
        axis=(1, 2),
        keepdims=True,
    )

    trial_z = (
        x - trial_median
    ) / (
        mad + 1e-8
    )

    artifact_ratio = np.mean(
        np.abs(trial_z) > 20.0
    )

    median_abs = float(
        np.median(np.abs(x))
    )

    valid_classes = float(
        np.sum(class_counts > 0)
        / CFG.n_classes
    )

    quality_score = (
        0.30 * valid_classes
        + 0.25 * class_balance
        + 0.20 * (1.0 - flat_ratio)
        + 0.20 * (1.0 - artifact_ratio)
        + 0.05 * min(
            np.log1p(len(y)) / np.log1p(120.0),
            1.0,
        )
    )

    return {
        "subject": subject_id,
        "valid": 1,
        "trials": len(y),
        "classes_present": int(
            np.sum(class_counts > 0)
        ),
        "class_balance": float(class_balance),
        "flat_ratio": float(flat_ratio),
        "artifact_ratio": float(artifact_ratio),
        "median_abs": median_abs,
        "quality_score": float(quality_score),
    }


def build_subject_quality_table():
    quality_file = (
        RESULT_ROOT
        / "subject_quality.csv"
    )

    if quality_file.exists():
        return pd.read_csv(
            quality_file
        )

    rows = []

    subjects_to_scan = (
        AVAILABLE_SUBJECTS
        if CFG.quality_scan_all_109
        else AVAILABLE_SUBJECTS[
            :CFG.n_source_subjects
            + CFG.n_target_subjects
            + 5
        ]
    )

    for subject_id in tqdm(
        subjects_to_scan,
        desc="Scanning subjects",
    ):
        rows.append(
            subject_quality_row(
                subject_id
            )
        )

    table = pd.DataFrame(rows)

    table = table.sort_values(
        "quality_score",
        ascending=False,
        ignore_index=True,
    )

    table.to_csv(
        quality_file,
        index=False,
    )

    return table


QUALITY_TABLE = build_subject_quality_table()

try:
    display(
        QUALITY_TABLE.head(20)
    )
except NameError:
    print(
        QUALITY_TABLE.head(20)
    )


def select_experiment_cohort():
    if CFG.subject_mode == "manual":
        source_subjects = [
            int(x)
            for x in CFG.manual_source_subjects
        ]

        target_subjects = [
            int(x)
            for x in CFG.manual_target_subjects
        ]

        return (
            source_subjects,
            target_subjects,
        )

    ranked = (
        QUALITY_TABLE["subject"]
        .astype(int)
        .tolist()
    )

    if CFG.subject_mode == "all_available":
        target_subjects = ranked[
            :CFG.n_target_subjects
        ]

        source_subjects = [
            s
            for s in ranked
            if s not in target_subjects
        ]

        return (
            source_subjects,
            target_subjects,
        )

    # Quality-ranked development cohort.
    source_subjects = ranked[
        :CFG.n_source_subjects
    ]

    remaining = [
        s
        for s in ranked
        if s not in source_subjects
    ]

    target_subjects = remaining[
        :CFG.n_target_subjects
    ]

    return (
        source_subjects,
        target_subjects,
    )


SOURCE_SUBJECTS, TARGET_SUBJECTS = (
    select_experiment_cohort()
)

print("\nDEVELOPMENT COHORT")
print("Source subjects:", SOURCE_SUBJECTS)
print("Target subjects:", TARGET_SUBJECTS)

with open(
    RESULT_ROOT / "cohort.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "source_subjects": SOURCE_SUBJECTS,
            "target_subjects": TARGET_SUBJECTS,
            "mode": CFG.subject_mode,
        },
        f,
        indent=2,
    )

,subject,valid,trials,classes_present,class_balance,flat_ratio,artifact_ratio,median_abs,quality_score
0,100,1,72,4,1.000000,0.0,0.000302,0.000025,0.994671
1,88,1,114,4,0.965517,0.0,0.000515,0.000031,0.990746
2,55,1,90,4,0.956522,0.0,0.000000,0.000012,0.986160
3,68,1,90,4,0.956522,0.0,0.000000,0.000056,0.986160
4,29,1,90,4,0.956522,0.0,0.000000,0.000016,0.986160
5,31,1,90,4,0.956522,0.0,0.000000,0.000024,0.986160
6,37,1,90,4,0.956522,0.0,0.000000,0.000008,0.986160
7,39,1,90,4,0.956522,0.0,0.000000,0.000069,0.986160
8,56,1,90,4,0.956522,0.0,0.000000,0.000037,0.986160
9,63,1,90,4,0.956522,0.0,0.000000,0.000051,0.986160



DEVELOPMENT COHORT
Source subjects: [100, 88, 55, 68, 29, 31, 37, 39, 56, 63, 64, 66, 73, 24, 76, 81, 84, 93, 95, 102]
Target subjects: [103, 28, 109, 11, 20]


In [4]:
# ============================================================
# CELL 5 - Preprocessing + FBCSP + LASSO
# ============================================================

def butter_bandpass(
    x,
    low,
    high,
    sfreq=CFG.sfreq,
    order=CFG.filter_order,
):
    nyquist = sfreq / 2.0

    low_n = max(
        low / nyquist,
        1e-5,
    )
    high_n = min(
        high / nyquist,
        0.999,
    )

    sos = butter(
        order,
        [low_n, high_n],
        btype="bandpass",
        output="sos",
    )

    return sosfiltfilt(
        sos,
        x,
        axis=-1,
    ).astype(np.float32)


def preprocess_training_fit(
    x_train,
):
    x_train = butter_bandpass(
        x_train,
        CFG.low_hz,
        CFG.high_hz,
    )

    mean = x_train.mean(
        axis=(0, 2),
        keepdims=True,
    )

    std = x_train.std(
        axis=(0, 2),
        keepdims=True,
    )

    std = np.maximum(
        std,
        1e-6,
    )

    return (
        ((x_train - mean) / std).astype(
            np.float32
        ),
        mean.astype(np.float32),
        std.astype(np.float32),
    )


def preprocess_apply(
    x,
    mean,
    std,
):
    x = butter_bandpass(
        x,
        CFG.low_hz,
        CFG.high_hz,
    )

    x = (
        (x - mean) / std
    ).astype(np.float32)

    return np.nan_to_num(
        x,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )


def normalized_covariance(trial):
    cov = trial @ trial.T
    return cov / (
        np.trace(cov)
        + 1e-8
    )


class OVRFBCSPLASSO:
    """
    FBCSP + LASSO implementation.

    10 bands × 4 OVR problems × 4 eigenvectors
    = 160 candidate features.
    """

    def __init__(self):
        self.filters = []
        self.band_indices = []
        self.feature_indices = None
        self.scaler = StandardScaler()

    def fit(self, x, y):
        self.filters = []
        self.band_indices = []

        for band_idx, (
            low,
            high,
        ) in enumerate(CFG.bands):

            x_band = butter_bandpass(
                x,
                low,
                high,
            )

            band_filters = []

            for class_id in range(
                CFG.n_classes
            ):
                x_pos = x_band[
                    y == class_id
                ]
                x_neg = x_band[
                    y != class_id
                ]

                if (
                    len(x_pos) == 0
                    or len(x_neg) == 0
                ):
                    continue

                r_pos = np.mean(
                    [
                        normalized_covariance(
                            trial
                        )
                        for trial in x_pos
                    ],
                    axis=0,
                )

                r_neg = np.mean(
                    [
                        normalized_covariance(
                            trial
                        )
                        for trial in x_neg
                    ],
                    axis=0,
                )

                composite = (
                    r_pos
                    + r_neg
                    + 1e-6 * np.eye(
                        CFG.channels
                    )
                )

                try:
                    eigvals, eigvecs = eigh(
                        r_pos,
                        composite,
                    )
                except Exception:
                    eigvals, eigvecs = np.linalg.eigh(
                        np.linalg.pinv(
                            composite
                        ) @ r_pos
                    )

                order = np.argsort(
                    eigvals
                )[::-1]

                selected = order[
                    :CFG.csp_vectors_per_ovr_class
                ]

                band_filters.append(
                    eigvecs[
                        :,
                        selected
                    ].T
                )

            band_filters = np.concatenate(
                band_filters,
                axis=0,
            )

            if band_filters.shape[0] != 16:
                raise RuntimeError(
                    "Expected 16 filters per band, "
                    f"got {band_filters.shape}"
                )

            self.filters.extend(
                list(band_filters)
            )

            self.band_indices.extend(
                [band_idx] * 16
            )

        self.filters = np.asarray(
            self.filters,
            dtype=np.float32,
        )

        self.band_indices = np.asarray(
            self.band_indices,
            dtype=np.int64,
        )

        features = self._raw_features(x)
        features_scaled = (
            self.scaler.fit_transform(
                features
            )
        )

        coefficients = []

        cv_splits = min(
            CFG.lasso_cv,
            int(
                np.min(
                    np.bincount(y)
                )
            ),
        )

        cv_splits = max(
            cv_splits,
            2,
        )

        for class_id in range(
            CFG.n_classes
        ):
            target = (
                y == class_id
            ).astype(np.float32)

            best_model = None
            best_error = np.inf

            skf = StratifiedKFold(
                n_splits=cv_splits,
                shuffle=True,
                random_state=SEED,
            )

            for alpha in CFG.lasso_alphas:
                errors = []

                for train_idx, val_idx in skf.split(
                    features_scaled,
                    target,
                ):
                    model = Lasso(
                        alpha=alpha,
                        max_iter=20000,
                        random_state=SEED,
                    )

                    model.fit(
                        features_scaled[
                            train_idx
                        ],
                        target[
                            train_idx
                        ],
                    )

                    pred = model.predict(
                        features_scaled[
                            val_idx
                        ]
                    )

                    errors.append(
                        np.mean(
                            (
                                pred
                                - target[
                                    val_idx
                                ]
                            )
                            ** 2
                        )
                    )

                error = float(
                    np.mean(errors)
                )

                if error < best_error:
                    best_error = error

                    best_model = Lasso(
                        alpha=alpha,
                        max_iter=20000,
                        random_state=SEED,
                    )

                    best_model.fit(
                        features_scaled,
                        target,
                    )

            coefficients.append(
                np.abs(
                    best_model.coef_
                )
            )

        coefficient_strength = np.sum(
            np.stack(coefficients),
            axis=0,
        )

        selected = np.flatnonzero(
            coefficient_strength > 1e-8
        )

        if (
            CFG.max_sparse_features is not None
            and len(selected)
            > CFG.max_sparse_features
        ):
            ranking = np.argsort(
                coefficient_strength[
                    selected
                ]
            )[::-1]

            selected = selected[
                ranking[
                    :CFG.max_sparse_features
                ]
            ]

        if len(selected) == 0:
            selected = np.argsort(
                coefficient_strength
            )[
                -min(
                    16,
                    len(coefficient_strength),
                ):
            ]

        self.feature_indices = np.sort(
            selected
        )

        print(
            "Candidate dimensions:",
            len(coefficient_strength),
        )
        print(
            "Selected sparse dimensions:",
            len(self.feature_indices),
        )

        return self

    def _raw_features(self, x):
        features = []

        for filter_idx, spatial_filter in enumerate(
            self.filters
        ):
            band_idx = int(
                self.band_indices[
                    filter_idx
                ]
            )

            low_hz, high_hz = (
                CFG.bands[band_idx]
            )

            x_band = butter_bandpass(
                x,
                low_hz,
                high_hz,
            )

            projected = np.einsum(
                "c,nct->nt",
                spatial_filter,
                x_band,
            )

            features.append(
                np.log(
                    np.var(
                        projected,
                        axis=1,
                    )
                    + 1e-8
                )
            )

        return np.stack(
            features,
            axis=1,
        ).astype(np.float32)

    def transform(self, x):
        features = self._raw_features(
            x
        )

        features = (
            self.scaler.transform(
                features
            )
        )

        return features[
            :,
            self.feature_indices,
        ].astype(np.float32)

    def selected_filters(self):
        return (
            self.filters[
                self.feature_indices
            ],
            self.band_indices[
                self.feature_indices
            ],
        )


print(
    "Expected candidate dimensions:",
    10
    * CFG.n_classes
    * CFG.csp_vectors_per_ovr_class,
)

Expected candidate dimensions: 160


In [5]:
# ============================================================
# CELL 6 - FBGAN: TARGET-SPECIFIC SPATIALLY CONSTRAINED GAN
#              MPS-SAFE VERSION
# ============================================================
#
# Purpose
# -------
# Target-subject FBGAN adaptation using:
#
#   z
#   ↓
# Generator
#   ↓
# synthetic EEG (B, 1, 64, 640)
#
# Two discriminators:
#
# D_phi
#   raw EEG
#   ↓
#   temporal processing
#   ↓
#   64-channel spatial collapse
#   ↓
#   temporal processing
#   ↓
#   MPS-safe global pooling
#   ↓
#   adversarial score
#
# D_psi
#   sparse CSP-filtered target representation
#   ↓
#   sparse-filter spatial collapse
#   ↓
#   temporal processing
#   ↓
#   MPS-safe global pooling
#   ↓
#   adversarial score
#
# IMPORTANT MPS FIX
# -----------------
# Apple MPS currently has a limitation for some
# AdaptiveAvgPool2d configurations when the input spatial
# dimension is not divisible by the requested output size.
#
# The previous implementation used:
#
#     nn.AdaptiveAvgPool2d((1, 25))
#
# and failed during:
#
#     D_phi(real)
#
# because the temporal dimension reaching the adaptive pooling
# layer was not divisible by 25.
#
# This cell replaces AdaptiveAvgPool2d with:
#
#     GlobalMeanPool2d()
#
# implemented with torch.mean(), which is MPS-safe.
#
# Expected dimensions
# -------------------
#
# Generator:
#
#     z                  (B, latent_dim)
#       ↓
#     FC
#       ↓
#     (B, 128, 8, 40)
#       ↓
#     transposed convs
#       ↓
#     interpolate
#       ↓
#     (B, 1, 64, 640)
#
# D_phi:
#
#     (B, 1, 64, 640)
#       ↓
#     temporal convolution
#       ↓
#     spatial convolution over 64 channels
#       ↓
#     temporal convolution / pooling
#       ↓
#     GlobalMeanPool2d
#       ↓
#     (B, 30)
#       ↓
#     (B, 1)
#
# D_psi:
#
#     (B, 1, N_sparse_filters, 640)
#       ↓
#     temporal convolution
#       ↓
#     spatial collapse across sparse CSP filters
#       ↓
#     temporal convolution / pooling
#       ↓
#     GlobalMeanPool2d
#       ↓
#     (B, 30)
#       ↓
#     (B, 1)
# ============================================================


class FBGANGenerator(nn.Module):
    """
    Target-specific EEG generator.

    Input:
        z -> (B, latent_dim)

    Output:
        fake EEG -> (B, 1, 64, 640)
    """

    def __init__(
        self,
        latent_dim=CFG.latent_dim,
    ):
        super().__init__()

        self.latent_dim = latent_dim

        # Compact replacement for the extremely large
        # 1600 -> 256000 FC used in the original geometry.
        #
        # The final EEG geometry remains exactly:
        #     (1, 64, 640)
        self.fc = nn.Linear(
            latent_dim,
            128 * 8 * 40,
        )

        self.layers = nn.Sequential(
            # ------------------------------------------------
            # Deconvolution 1
            # ------------------------------------------------
            nn.ConvTranspose2d(
                128,
                128,
                kernel_size=(3, 15),
                stride=(1, 3),
                padding=(1, 7),
            ),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Deconvolution 2
            # ------------------------------------------------
            nn.ConvTranspose2d(
                128,
                128,
                kernel_size=(3, 15),
                stride=(1, 3),
                padding=(1, 7),
            ),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Deconvolution 3
            # ------------------------------------------------
            nn.ConvTranspose2d(
                128,
                64,
                kernel_size=(3, 5),
                stride=(1, 2),
                padding=(1, 2),
            ),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Deconvolution 4
            # ------------------------------------------------
            nn.ConvTranspose2d(
                64,
                32,
                kernel_size=(4, 5),
                stride=(2, 1),
                padding=(1, 2),
            ),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Output layer
            # ------------------------------------------------
            nn.ConvTranspose2d(
                32,
                1,
                kernel_size=(1, 2),
                stride=(1, 1),
            ),
        )

    def forward(
        self,
        z,
        debug=False,
    ):
        if z.ndim != 2:
            raise ValueError(
                "FBGANGenerator expected latent tensor "
                "with shape (B, latent_dim). "
                f"Received {tuple(z.shape)}."
            )

        if debug:
            print(
                "Generator latent:",
                tuple(z.shape),
            )

        x = self.fc(z)

        x = x.view(
            -1,
            128,
            8,
            40,
        )

        if debug:
            print(
                "Generator reshape:",
                tuple(x.shape),
            )

        x = self.layers(x)

        if debug:
            print(
                "Generator deconv output:",
                tuple(x.shape),
            )

        # Exact PhysioNet target geometry:
        #
        #     channels = 64
        #     samples  = 640
        #
        # interpolate() is supported on MPS and avoids geometry
        # assumptions about the preceding deconvolution stack.
        x = F.interpolate(
            x,
            size=(
                CFG.channels,
                CFG.samples,
            ),
            mode="bilinear",
            align_corners=False,
        )

        x = torch.tanh(x)

        if debug:
            print(
                "Generator final output:",
                tuple(x.shape),
            )

        return x


class GlobalMeanPool2d(nn.Module):
    """
    MPS-safe global mean pooling.

    Input:
        (B, C, H, W)

    Output:
        (B, C, 1, 1)

    This replaces AdaptiveAvgPool2d((1, 25)).
    """

    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x.mean(
            dim=(-2, -1),
            keepdim=True,
        )


class Dphi(nn.Module):
    """
    Raw EEG discriminator.

    Input:
        (B, 1, 64, 640)

    Output:
        (B, 1)
    """

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            # ------------------------------------------------
            # Temporal convolution
            # ------------------------------------------------
            nn.Conv2d(
                in_channels=1,
                out_channels=10,
                kernel_size=(1, 23),
                stride=1,
                padding=0,
                bias=True,
            ),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Spatial convolution across all 64 EEG channels
            # ------------------------------------------------
            nn.Conv2d(
                in_channels=10,
                out_channels=30,
                kernel_size=(
                    CFG.channels,
                    1,
                ),
                stride=1,
                padding=0,
                bias=True,
            ),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Temporal convolution
            # ------------------------------------------------
            nn.Conv2d(
                in_channels=30,
                out_channels=30,
                kernel_size=(1, 17),
                stride=1,
                padding=0,
                bias=True,
            ),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Temporal pooling
            # ------------------------------------------------
            nn.MaxPool2d(
                kernel_size=(1, 6),
                stride=(1, 6),
            ),

            # ------------------------------------------------
            # Additional temporal convolution
            # ------------------------------------------------
            nn.Conv2d(
                in_channels=30,
                out_channels=30,
                kernel_size=(1, 7),
                stride=1,
                padding=0,
                bias=True,
            ),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Additional temporal pooling
            # ------------------------------------------------
            nn.MaxPool2d(
                kernel_size=(1, 6),
                stride=(1, 6),
            ),

            # ------------------------------------------------
            # MPS-SAFE REPLACEMENT
            # ------------------------------------------------
            GlobalMeanPool2d(),
        )

        self.fc = nn.Linear(
            in_features=30,
            out_features=1,
            bias=True,
        )

    def forward(
        self,
        x,
        debug=False,
    ):
        if x.ndim != 4:
            raise ValueError(
                "D_phi expected input shape "
                "(B, 1, 64, 640). "
                f"Received {tuple(x.shape)}."
            )

        if x.shape[1] != 1:
            raise ValueError(
                "D_phi expects one EEG input channel. "
                f"Received {x.shape[1]}."
            )

        if x.shape[2] != CFG.channels:
            raise ValueError(
                "D_phi channel dimension mismatch. "
                f"Expected {CFG.channels}, "
                f"received {x.shape[2]}."
            )

        if x.shape[3] != CFG.samples:
            raise ValueError(
                "D_phi time dimension mismatch. "
                f"Expected {CFG.samples}, "
                f"received {x.shape[3]}."
            )

        if debug:
            print(
                "D_phi input:",
                tuple(x.shape),
            )

        x = self.net(x)

        if debug:
            print(
                "D_phi pooled:",
                tuple(x.shape),
            )

        x = x.flatten(1)

        if debug:
            print(
                "D_phi flattened:",
                tuple(x.shape),
            )

        x = self.fc(x)

        if debug:
            print(
                "D_phi output:",
                tuple(x.shape),
            )

        return x


class Dpsi(nn.Module):
    """
    Sparse CSP/filter-bank discriminator.

    Input:
        (B, 1, N_sparse_filters, 640)

    For the current experiment:
        N_sparse_filters = 64

    Output:
        (B, 1)
    """

    def __init__(
        self,
        n_sparse_filters,
    ):
        super().__init__()

        self.n_sparse_filters = int(
            n_sparse_filters
        )

        if self.n_sparse_filters <= 0:
            raise ValueError(
                "n_sparse_filters must be positive."
            )

        self.net = nn.Sequential(
            # ------------------------------------------------
            # Temporal convolution
            # ------------------------------------------------
            nn.Conv2d(
                in_channels=1,
                out_channels=10,
                kernel_size=(1, 23),
                stride=1,
                padding=0,
                bias=True,
            ),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Spatial collapse across selected CSP filters
            # ------------------------------------------------
            nn.Conv2d(
                in_channels=10,
                out_channels=30,
                kernel_size=(
                    self.n_sparse_filters,
                    1,
                ),
                stride=1,
                padding=0,
                bias=True,
            ),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Temporal convolution
            # ------------------------------------------------
            nn.Conv2d(
                in_channels=30,
                out_channels=30,
                kernel_size=(1, 17),
                stride=1,
                padding=0,
                bias=True,
            ),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Temporal pooling
            # ------------------------------------------------
            nn.MaxPool2d(
                kernel_size=(1, 6),
                stride=(1, 6),
            ),

            # ------------------------------------------------
            # Additional temporal convolution
            # ------------------------------------------------
            nn.Conv2d(
                in_channels=30,
                out_channels=30,
                kernel_size=(1, 7),
                stride=1,
                padding=0,
                bias=True,
            ),
            nn.LeakyReLU(
                0.2,
                inplace=True,
            ),

            # ------------------------------------------------
            # Additional temporal pooling
            # ------------------------------------------------
            nn.MaxPool2d(
                kernel_size=(1, 6),
                stride=(1, 6),
            ),

            # ------------------------------------------------
            # MPS-safe global pooling
            # ------------------------------------------------
            GlobalMeanPool2d(),
        )

        self.fc = nn.Linear(
            in_features=30,
            out_features=1,
            bias=True,
        )

    def forward(
        self,
        x,
        debug=False,
    ):
        if x.ndim != 4:
            raise ValueError(
                "D_psi expected input shape "
                "(B, 1, N_sparse_filters, 640). "
                f"Received {tuple(x.shape)}."
            )

        if x.shape[1] != 1:
            raise ValueError(
                "D_psi expects one input channel. "
                f"Received {x.shape[1]}."
            )

        if x.shape[2] != self.n_sparse_filters:
            raise ValueError(
                "D_psi sparse-filter dimension mismatch. "
                f"Expected {self.n_sparse_filters}, "
                f"received {x.shape[2]}."
            )

        if x.shape[3] != CFG.samples:
            raise ValueError(
                "D_psi time dimension mismatch. "
                f"Expected {CFG.samples}, "
                f"received {x.shape[3]}."
            )

        if debug:
            print(
                "D_psi input:",
                tuple(x.shape),
            )

        x = self.net(x)

        if debug:
            print(
                "D_psi pooled:",
                tuple(x.shape),
            )

        x = x.flatten(1)

        if debug:
            print(
                "D_psi flattened:",
                tuple(x.shape),
            )

        x = self.fc(x)

        if debug:
            print(
                "D_psi output:",
                tuple(x.shape),
            )

        return x


# ============================================================
# DIFFERENTIABLE FIR FILTER FOR D_PSI
# ============================================================


@lru_cache(maxsize=32)
def build_fir_kernel(
    low_hz,
    high_hz,
    fs=CFG.sfreq,
):
    """
    Build a fixed FIR kernel for differentiable
    filter-bank processing inside D_psi.

    The FIR coefficients are fixed.
    Gradients can still pass from D_psi into the
    generator through torch.conv1d.
    """

    coeffs = mne.filter.create_filter(
        np.zeros(
            (1, 4096)
        ),
        sfreq=fs,
        l_freq=low_hz,
        h_freq=high_hz,
        method="fir",
        fir_design="firwin",
        phase="zero-double",
        verbose=False,
    )

    coeffs = np.asarray(
        coeffs,
        dtype=np.float32,
    )

    # Keep the filter compact enough for MacBook/MPS execution.
    max_taps = 129

    if coeffs.size > max_taps:
        center = (
            coeffs.size // 2
        )

        half = (
            max_taps // 2
        )

        coeffs = coeffs[
            center - half:
            center + half + 1
        ]

    coeffs = coeffs / (
        np.sum(
            np.abs(coeffs)
        )
        + 1e-8
    )

    return coeffs.astype(
        np.float32
    )


def torch_bandpass_filter(
    x,
    low_hz,
    high_hz,
):
    """
    Differentiable FIR band-pass filtering.

    Input:
        (B, C, T)

    Output:
        (B, C, T)
    """

    kernel = build_fir_kernel(
        low_hz,
        high_hz,
    )

    kernel = torch.tensor(
        kernel,
        dtype=x.dtype,
        device=x.device,
    ).view(
        1,
        1,
        -1,
    )

    batch_size, channels, time_points = (
        x.shape
    )

    x_reshape = x.reshape(
        batch_size * channels,
        1,
        time_points,
    )

    padding = (
        kernel.shape[-1] // 2
    )

    filtered = F.conv1d(
        x_reshape,
        kernel,
        padding=padding,
    )

    # Defensive correction if an even filter length produces
    # a one-sample discrepancy.
    if filtered.shape[-1] != time_points:

        filtered = F.interpolate(
            filtered,
            size=time_points,
            mode="linear",
            align_corners=False,
        )

    return filtered.reshape(
        batch_size,
        channels,
        time_points,
    )


def sparse_fb_features_torch(
    x,
    selected_filters,
    selected_band_indices,
):
    """
    Differentiable sparse FBCSP projection.

    Input:
        x -> (B, 1, 64, 640)

    Output:
        (B, N_selected_filters, 640)
    """

    if x.ndim != 4:
        raise ValueError(
            "sparse_fb_features_torch expected "
            "(B, 1, C, T). "
            f"Received {tuple(x.shape)}."
        )

    x_input = x.squeeze(1)

    if x_input.shape[1] != CFG.channels:
        raise ValueError(
            "Sparse FB input channel mismatch."
        )

    outputs = []

    unique_bands = sorted(
        set(
            int(v)
            for v in selected_band_indices
        )
    )

    band_cache = {}

    for band_idx in unique_bands:

        low_hz, high_hz = (
            CFG.bands[
                band_idx
            ]
        )

        band_cache[band_idx] = (
            torch_bandpass_filter(
                x_input,
                low_hz,
                high_hz,
            )
        )

    filters_tensor = torch.tensor(
        selected_filters,
        dtype=x.dtype,
        device=x.device,
    )

    for filter_idx in range(
        len(selected_filters)
    ):

        band_idx = int(
            selected_band_indices[
                filter_idx
            ]
        )

        x_band = (
            band_cache[band_idx]
        )

        spatial_filter = (
            filters_tensor[
                filter_idx
            ]
        )

        projected = torch.einsum(
            "c,bct->bt",
            spatial_filter,
            x_band,
        )

        outputs.append(
            projected
        )

    return torch.stack(
        outputs,
        dim=1,
    )


# ============================================================
# TRAIN ONE TARGET-CLASS FBGAN
# ============================================================


def train_one_fbgan_class(
    target_x_class,
    selected_filters,
    selected_band_indices,
    epochs=CFG.gan_epochs,
    debug=False,
):
    """
    Train FBGAN using only target calibration trials
    for one motor-imagery class.

    target_x_class:
        (N_calibration, 64, 640)

    selected_filters:
        sparse CSP filters

    selected_band_indices:
        corresponding filter-bank indices
    """

    if target_x_class.ndim != 3:
        raise ValueError(
            "target_x_class must have shape "
            "(N, 64, 640). "
            f"Received {target_x_class.shape}."
        )

    if target_x_class.shape[1:] != (
        CFG.channels,
        CFG.samples,
    ):
        raise ValueError(
            "Target calibration geometry mismatch. "
            f"Expected ({CFG.channels}, {CFG.samples}), "
            f"received {target_x_class.shape[1:]}."
        )

    if len(target_x_class) < 2:
        raise ValueError(
            "At least two calibration trials are "
            "required for FBGAN training."
        )

    # Scale target EEG to a stable GAN range.
    target_x_class = np.clip(
        target_x_class / CFG.gan_clip,
        -1.0,
        1.0,
    ).astype(
        np.float32
    )

    loader = DataLoader(
        TensorEEGDataset(
            target_x_class
        ),
        batch_size=CFG.gan_batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=CFG.num_workers,
    )

    # --------------------------------------------------------
    # Networks
    # --------------------------------------------------------

    generator = FBGANGenerator().to(
        DEVICE
    )

    d_phi = Dphi().to(
        DEVICE
    )

    d_psi = Dpsi(
        n_sparse_filters=len(
            selected_filters
        )
    ).to(
        DEVICE
    )

    # --------------------------------------------------------
    # Optimizers
    # --------------------------------------------------------

    opt_g = torch.optim.Adam(
        generator.parameters(),
        lr=CFG.gan_lr,
        betas=(0.5, 0.999),
    )

    opt_phi = torch.optim.Adam(
        d_phi.parameters(),
        lr=CFG.gan_lr,
        betas=(0.5, 0.999),
    )

    opt_psi = torch.optim.Adam(
        d_psi.parameters(),
        lr=CFG.gan_lr,
        betas=(0.5, 0.999),
    )

    criterion = nn.BCEWithLogitsLoss()

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(
        1,
        epochs + 1,
    ):

        g_values = []
        d_values = []

        for real in loader:

            real = (
                real
                .unsqueeze(1)
                .to(DEVICE)
            )

            batch_size = (
                real.shape[0]
            )

            real_labels = (
                torch.ones(
                    batch_size,
                    1,
                    device=DEVICE,
                )
            )

            fake_labels = (
                torch.zeros(
                    batch_size,
                    1,
                    device=DEVICE,
                )
            )

            # =================================================
            # DISCRIMINATOR STEP
            # =================================================

            opt_phi.zero_grad(
                set_to_none=True
            )

            opt_psi.zero_grad(
                set_to_none=True
            )

            z = torch.randn(
                batch_size,
                CFG.latent_dim,
                device=DEVICE,
            )

            with torch.no_grad():

                fake_detached = (
                    generator(z)
                )

            # ------------------------------------------------
            # Raw EEG discriminator
            # ------------------------------------------------

            phi_real = d_phi(
                real,
                debug=(
                    debug
                    and epoch == 1
                ),
            )

            phi_fake = d_phi(
                fake_detached
            )

            # ------------------------------------------------
            # Sparse FBCSP discriminator
            # ------------------------------------------------

            real_sparse = (
                sparse_fb_features_torch(
                    real,
                    selected_filters,
                    selected_band_indices,
                )
                .unsqueeze(1)
            )

            fake_sparse = (
                sparse_fb_features_torch(
                    fake_detached,
                    selected_filters,
                    selected_band_indices,
                )
                .unsqueeze(1)
            )

            if (
                debug
                and epoch == 1
            ):
                print(
                    "D_psi sparse real:",
                    tuple(
                        real_sparse.shape
                    ),
                )

                print(
                    "D_psi sparse fake:",
                    tuple(
                        fake_sparse.shape
                    ),
                )

            psi_real = d_psi(
                real_sparse,
                debug=(
                    debug
                    and epoch == 1
                ),
            )

            psi_fake = d_psi(
                fake_sparse
            )

            # ------------------------------------------------
            # Discriminator loss
            # ------------------------------------------------

            d_loss_phi = (
                criterion(
                    phi_real,
                    real_labels,
                )
                + criterion(
                    phi_fake,
                    fake_labels,
                )
            )

            d_loss_psi = (
                criterion(
                    psi_real,
                    real_labels,
                )
                + criterion(
                    psi_fake,
                    fake_labels,
                )
            )

            d_loss = (
                d_loss_phi
                + d_loss_psi
            )

            d_loss.backward()

            opt_phi.step()
            opt_psi.step()

            # =================================================
            # GENERATOR STEP
            # =================================================

            opt_g.zero_grad(
                set_to_none=True
            )

            z = torch.randn(
                batch_size,
                CFG.latent_dim,
                device=DEVICE,
            )

            fake = generator(
                z,
                debug=(
                    debug
                    and epoch == 1
                ),
            )

            phi_fake = d_phi(
                fake
            )

            fake_sparse = (
                sparse_fb_features_torch(
                    fake,
                    selected_filters,
                    selected_band_indices,
                )
                .unsqueeze(1)
            )

            psi_fake = d_psi(
                fake_sparse
            )

            g_loss_phi = criterion(
                phi_fake,
                real_labels,
            )

            g_loss_psi = criterion(
                psi_fake,
                real_labels,
            )

            g_loss = (
                g_loss_phi
                + g_loss_psi
            )

            g_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                generator.parameters(),
                max_norm=1.0,
            )

            opt_g.step()

            g_values.append(
                float(
                    g_loss.detach()
                    .cpu()
                )
            )

            d_values.append(
                float(
                    d_loss.detach()
                    .cpu()
                )
            )

        # ----------------------------------------------------
        # Epoch logging
        # ----------------------------------------------------

        if (
            epoch == 1
            or epoch % 5 == 0
            or epoch == epochs
        ):

            mean_g = (
                float(
                    np.mean(
                        g_values
                    )
                )
                if g_values
                else float("nan")
            )

            mean_d = (
                float(
                    np.mean(
                        d_values
                    )
                )
                if d_values
                else float("nan")
            )

            print(
                f"GAN {epoch:03d}/{epochs} | "
                f"G={mean_g:.4f} | "
                f"D={mean_d:.4f}"
            )

    # Return on CPU to reduce persistent MPS memory.
    return generator.cpu()


# ============================================================
# GENERATE SYNTHETIC EEG
# ============================================================


def generate_fbgan_samples(
    generator,
    n_samples,
):
    """
    Generate target-specific synthetic EEG.

    Output:
        (N, 64, 640)
    """

    if n_samples <= 0:
        raise ValueError(
            "n_samples must be positive."
        )

    generator = generator.to(
        DEVICE
    )

    generator.eval()

    outputs = []

    with torch.no_grad():

        for start in range(
            0,
            n_samples,
            32,
        ):

            count = min(
                32,
                n_samples - start,
            )

            z = torch.randn(
                count,
                CFG.latent_dim,
                device=DEVICE,
            )

            fake = generator(
                z
            )

            if fake.shape[1:] != (
                1,
                CFG.channels,
                CFG.samples,
            ):
                raise RuntimeError(
                    "FBGAN generated an incorrect "
                    "EEG shape. Expected "
                    f"(B, 1, {CFG.channels}, "
                    f"{CFG.samples}), got "
                    f"{tuple(fake.shape)}."
                )

            fake = (
                fake
                .cpu()
                .numpy()
                .squeeze(1)
            )

            fake = (
                fake
                * CFG.gan_clip
            )

            outputs.append(
                fake.astype(
                    np.float32
                )
            )

    return np.concatenate(
        outputs,
        axis=0,
    )


# ============================================================
# EEG DATASET FOR FBGAN
# ============================================================


class TensorEEGDataset(
    Dataset
):
    """
    Simple EEG Tensor dataset.

    Input:
        (N, 64, 640)
    """

    def __init__(
        self,
        x,
    ):
        self.x = torch.tensor(
            x,
            dtype=torch.float32,
        )

    def __len__(self):
        return len(
            self.x
        )

    def __getitem__(
        self,
        idx,
    ):
        return self.x[idx]


# ============================================================
# COMPLETE MPS SHAPE TEST
# ============================================================


def test_fbgan_mps_shapes():
    """
    Complete forward-shape test.

    This MUST pass before starting the full LOSO experiment.
    """

    print("=" * 78)
    print(
        "FBGAN MPS SHAPE TEST"
    )
    print("=" * 78)

    print(
        "Device:",
        DEVICE,
    )

    # --------------------------------------------------------
    # Generator
    # --------------------------------------------------------

    generator = (
        FBGANGenerator()
        .to(DEVICE)
    )

    z = torch.randn(
        2,
        CFG.latent_dim,
        device=DEVICE,
    )

    with torch.no_grad():

        fake = generator(
            z,
            debug=True,
        )

    expected_generator_shape = (
        2,
        1,
        CFG.channels,
        CFG.samples,
    )

    assert (
        tuple(fake.shape)
        == expected_generator_shape
    ), (
        "Generator shape mismatch: "
        f"expected "
        f"{expected_generator_shape}, "
        f"got {tuple(fake.shape)}"
    )

    # --------------------------------------------------------
    # D_phi
    # --------------------------------------------------------

    d_phi = (
        Dphi()
        .to(DEVICE)
    )

    raw_input = torch.randn(
        2,
        1,
        CFG.channels,
        CFG.samples,
        device=DEVICE,
    )

    with torch.no_grad():

        phi_output = d_phi(
            raw_input,
            debug=True,
        )

    assert tuple(
        phi_output.shape
    ) == (
        2,
        1,
    ), (
        "D_phi output mismatch: "
        f"got {tuple(phi_output.shape)}"
    )

    # --------------------------------------------------------
    # D_psi
    # --------------------------------------------------------

    n_sparse_filters = min(
        CFG.max_sparse_features,
        64,
    )

    d_psi = (
        Dpsi(
            n_sparse_filters
        )
        .to(DEVICE)
    )

    sparse_input = torch.randn(
        2,
        1,
        n_sparse_filters,
        CFG.samples,
        device=DEVICE,
    )

    with torch.no_grad():

        psi_output = d_psi(
            sparse_input,
            debug=True,
        )

    assert tuple(
        psi_output.shape
    ) == (
        2,
        1,
    ), (
        "D_psi output mismatch: "
        f"got {tuple(psi_output.shape)}"
    )

    # --------------------------------------------------------
    # Final confirmation
    # --------------------------------------------------------

    print()
    print(
        "Generator:",
        tuple(fake.shape),
    )

    print(
        "D_phi:",
        tuple(phi_output.shape),
    )

    print(
        "D_psi:",
        tuple(psi_output.shape),
    )

    print()
    print("=" * 78)
    print(
        "FBGAN MPS SHAPE TEST PASSED"
    )
    print("=" * 78)


# ============================================================
# RUN SHAPE TEST
# ============================================================

test_fbgan_mps_shapes()

FBGAN MPS SHAPE TEST
Device: mps
Generator latent: (2, 256)
Generator reshape: (2, 128, 8, 40)
Generator deconv output: (2, 1, 16, 704)
Generator final output: (2, 1, 64, 640)
D_phi input: (2, 1, 64, 640)
D_phi pooled: (2, 30, 1, 1)
D_phi flattened: (2, 30)
D_phi output: (2, 1)
D_psi input: (2, 1, 64, 640)
D_psi pooled: (2, 30, 1, 1)
D_psi flattened: (2, 30)
D_psi output: (2, 1)

Generator: (2, 1, 64, 640)
D_phi: (2, 1)
D_psi: (2, 1)

FBGAN MPS SHAPE TEST PASSED


In [6]:
# ============================================================
# CELL 7 - Spatial-Temporal Transformer Classifier
# ============================================================


class SpatialTemporalTransformer(
    nn.Module
):
    """
    EEG-Conformer-inspired compact front-end:

    raw EEG
        -> temporal convolution
        -> depthwise spatial convolution
        -> separable temporal convolution
        -> temporal tokenization
        -> Transformer Encoder
        -> discriminative feature
        -> class logits
    """

    def __init__(
        self,
        n_classes=CFG.n_classes,
        embed_dim=CFG.embed_dim,
    ):
        super().__init__()

        self.temporal_conv = nn.Conv2d(
            1,
            16,
            kernel_size=(1, 31),
            padding=(0, 15),
            bias=False,
        )

        self.temporal_bn = nn.BatchNorm2d(
            16
        )

        self.spatial_conv = nn.Conv2d(
            16,
            16,
            kernel_size=(
                CFG.channels,
                1,
            ),
            groups=16,
            bias=False,
        )

        self.spatial_bn = nn.BatchNorm2d(
            16
        )

        self.separable_depthwise = nn.Conv2d(
            16,
            16,
            kernel_size=(1, 15),
            padding=(0, 7),
            groups=16,
            bias=False,
        )

        self.separable_pointwise = nn.Conv2d(
            16,
            embed_dim,
            kernel_size=1,
            bias=False,
        )

        self.feature_bn = nn.BatchNorm2d(
            embed_dim
        )

        self.pool = nn.AvgPool2d(
            kernel_size=(1, 4),
            stride=(1, 4),
        )

        self.token_pool = nn.AdaptiveAvgPool2d(
            (1, 40)
        )

        self.cls_token = nn.Parameter(
            torch.zeros(
                1,
                1,
                embed_dim,
            )
        )

        self.positional = nn.Parameter(
            torch.zeros(
                1,
                41,
                embed_dim,
            )
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=CFG.n_heads,
                dim_feedforward=CFG.ff_dim,
                dropout=CFG.transformer_dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
        )

        self.transformer = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=CFG.n_transformer_layers,
                enable_nested_tensor=False,
            )
        )

        self.feature_norm = nn.LayerNorm(
            embed_dim
        )

        self.dropout = nn.Dropout(
            CFG.transformer_dropout
        )

        self.classifier = nn.Linear(
            embed_dim,
            n_classes,
        )

        nn.init.trunc_normal_(
            self.cls_token,
            std=0.02,
        )

        nn.init.trunc_normal_(
            self.positional,
            std=0.02,
        )

    def forward(
        self,
        x,
        debug=False,
    ):
        if x.ndim == 3:
            x = x.unsqueeze(1)

        if debug:
            print(
                "Input:",
                x.shape,
            )

        x = self.temporal_conv(x)

        if debug:
            print(
                "Temporal:",
                x.shape,
            )

        x = F.gelu(
            self.temporal_bn(x)
        )

        x = self.spatial_conv(x)

        if debug:
            print(
                "Spatial:",
                x.shape,
            )

        x = F.gelu(
            self.spatial_bn(x)
        )

        x = self.pool(x)

        if debug:
            print(
                "Pool:",
                x.shape,
            )

        x = self.separable_depthwise(x)
        x = self.separable_pointwise(x)

        if debug:
            print(
                "Embedding:",
                x.shape,
            )

        x = F.gelu(
            self.feature_bn(x)
        )

        x = self.token_pool(x)

        if debug:
            print(
                "Token map:",
                x.shape,
            )

        x = x.squeeze(2).transpose(
            1,
            2,
        )

        batch_size = x.shape[0]

        cls = self.cls_token.expand(
            batch_size,
            -1,
            -1,
        )

        x = torch.cat(
            [cls, x],
            dim=1,
        )

        x = x + self.positional[
            :,
            :x.shape[1],
            :,
        ]

        if debug:
            print(
                "Tokens:",
                x.shape,
            )

        x = self.transformer(x)

        feature = self.feature_norm(
            x[:, 0]
        )

        logits = self.classifier(
            self.dropout(feature)
        )

        if debug:
            print(
                "Feature:",
                feature.shape,
            )

            print(
                "Logits:",
                logits.shape,
            )

        return (
            logits,
            feature,
        )


model = SpatialTemporalTransformer().to(
    DEVICE
)

with torch.no_grad():
    dummy = torch.randn(
        2,
        1,
        CFG.channels,
        CFG.samples,
        device=DEVICE,
    )

    logits, feature = model(
        dummy,
        debug=CFG.debug_shapes,
    )

assert logits.shape == (
    2,
    CFG.n_classes,
)

assert feature.shape == (
    2,
    CFG.embed_dim,
)

print(
    "Transformer shape test passed:",
    logits.shape,
    feature.shape,
)

Transformer shape test passed: torch.Size([2, 4]) torch.Size([2, 64])


In [7]:
# ============================================================
# CELL 8 - Center loss, CORAL, datasets, training utilities
# ============================================================


class DiscriminativeCenterLoss(
    nn.Module
):
    def __init__(
        self,
        n_classes,
        feature_dim,
    ):
        super().__init__()

        self.register_buffer(
            "centers",
            torch.zeros(
                n_classes,
                feature_dim,
            ),
        )

    @torch.no_grad()
    def initialize_centers(
        self,
        features,
        labels,
    ):
        for class_id in range(
            self.centers.shape[0]
        ):
            mask = (
                labels == class_id
            )

            if mask.any():
                self.centers[
                    class_id
                ] = features[
                    mask
                ].mean(dim=0)

    def forward(
        self,
        features,
        labels,
    ):
        centers = self.centers[
            labels
        ]

        return torch.norm(
            features - centers,
            p=2,
            dim=1,
        ).mean()

    @torch.no_grad()
    def center_shift(
        self,
        alpha=CFG.center_alpha,
    ):
        global_center = self.centers.mean(
            dim=0,
            keepdim=True,
        )

        self.centers.add_(
            alpha
            * (
                self.centers
                - global_center
            )
        )


def coral_loss(
    source_features,
    target_features,
):
    if (
        source_features.shape[0] < 2
        or target_features.shape[0] < 2
    ):
        return torch.zeros(
            (),
            device=source_features.device,
        )

    source = (
        source_features
        - source_features.mean(
            dim=0,
            keepdim=True,
        )
    )

    target = (
        target_features
        - target_features.mean(
            dim=0,
            keepdim=True,
        )
    )

    cov_source = (
        source.T @ source
        / max(
            source.shape[0] - 1,
            1,
        )
    )

    cov_target = (
        target.T @ target
        / max(
            target.shape[0] - 1,
            1,
        )
    )

    return (
        (cov_source - cov_target)
        .pow(2)
        .mean()
    )


class DomainEEGDataset(
    Dataset
):
    def __init__(
        self,
        x,
        y,
        domain,
    ):
        self.x = torch.tensor(
            x,
            dtype=torch.float32,
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long,
        )

        self.domain = torch.tensor(
            domain,
            dtype=torch.long,
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return (
            self.x[index],
            self.y[index],
            self.domain[index],
        )


def create_balanced_loader(
    x,
    y,
    domain,
    batch_size=CFG.batch_size,
):
    class_count = np.bincount(
        y,
        minlength=CFG.n_classes,
    )

    weights_by_class = (
        1.0
        / np.maximum(
            class_count,
            1,
        )
    )

    sample_weights = (
        weights_by_class[y]
    )

    sampler = WeightedRandomSampler(
        weights=torch.tensor(
            sample_weights,
            dtype=torch.double,
        ),
        num_samples=len(y),
        replacement=True,
    )

    dataset = DomainEEGDataset(
        x,
        y,
        domain,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=CFG.num_workers,
    )


def forward_features(
    model,
    x,
    batch_size=64,
):
    model.eval()

    outputs = []

    loader = DataLoader(
        TensorEEGDataset(x),
        batch_size=batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
    )

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(
                DEVICE
            )

            _, feature = model(
                batch
            )

            outputs.append(
                feature.cpu()
            )

    return torch.cat(
        outputs,
        dim=0,
    )


def initialize_center_vectors(
    model,
    center_loss,
    x,
    y,
):
    features = (
        forward_features(
            model,
            x,
        ).to(DEVICE)
    )

    labels = torch.tensor(
        y,
        dtype=torch.long,
        device=DEVICE,
    )

    center_loss.initialize_centers(
        features,
        labels,
    )


def train_supervised_stage(
    model,
    center_loss,
    train_x,
    train_y,
    validation_x,
    validation_y,
    epochs,
    lr,
    stage_name,
):
    domain = np.zeros(
        len(train_y),
        dtype=np.int64,
    )

    loader = create_balanced_loader(
        train_x,
        train_y,
        domain,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=CFG.weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=5,
        )
    )

    best_accuracy = -np.inf
    best_state = None
    patience_counter = 0

    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):
        model.train()

        batch_losses = []
        train_predictions = []
        train_targets = []

        for (
            x_batch,
            y_batch,
            _,
        ) in loader:

            x_batch = x_batch.to(
                DEVICE
            )

            y_batch = y_batch.to(
                DEVICE
            )

            optimizer.zero_grad()

            logits, features = model(
                x_batch
            )

            ce = F.cross_entropy(
                logits,
                y_batch,
            )

            center = center_loss(
                features,
                y_batch,
            )

            loss = (
                ce
                + CFG.lambda_center
                * center
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            optimizer.step()

            batch_losses.append(
                float(
                    loss.detach().cpu()
                )
            )

            train_predictions.extend(
                logits.argmax(
                    dim=1
                )
                .detach()
                .cpu()
                .numpy()
            )

            train_targets.extend(
                y_batch.detach()
                .cpu()
                .numpy()
            )

        if (
            epoch
            % CFG.center_update_every
            == 0
        ):
            center_loss.center_shift(
                CFG.center_alpha
            )

        model.eval()
        val_predictions = []

        with torch.no_grad():
            val_loader = DataLoader(
                TensorEEGDataset(
                    validation_x
                ),
                batch_size=64,
                shuffle=False,
            )

            for batch in val_loader:
                batch = batch.to(
                    DEVICE
                )

                logits, _ = model(
                    batch
                )

                val_predictions.extend(
                    logits.argmax(
                        dim=1
                    )
                    .cpu()
                    .numpy()
                )

        val_accuracy = accuracy_score(
            validation_y,
            val_predictions,
        )

        train_accuracy = accuracy_score(
            train_targets,
            train_predictions,
        )

        scheduler.step(
            val_accuracy
        )

        history.append(
            {
                "epoch": epoch,
                "stage": stage_name,
                "train_loss": float(
                    np.mean(
                        batch_losses
                    )
                ),
                "train_accuracy": train_accuracy,
                "validation_accuracy": val_accuracy,
                "lr": optimizer.param_groups[
                    0
                ]["lr"],
            }
        )

        if (
            val_accuracy
            > best_accuracy
        ):
            best_accuracy = val_accuracy

            best_state = {
                key: value.detach()
                .cpu()
                .clone()
                for key, value
                in model.state_dict()
                .items()
            }

            patience_counter = 0
        else:
            patience_counter += 1

        if (
            epoch == 1
            or epoch % 5 == 0
        ):
            print(
                f"[{stage_name}] "
                f"{epoch:03d}/{epochs} | "
                f"loss={np.mean(batch_losses):.4f} | "
                f"train={train_accuracy:.4f} | "
                f"val={val_accuracy:.4f}"
            )

        if (
            patience_counter
            >= CFG.early_stopping_patience
        ):
            print(
                f"[{stage_name}] early stopping."
            )
            break

    if best_state is not None:
        model.load_state_dict(
            best_state
        )

    return (
        model,
        pd.DataFrame(history),
    )


def train_adaptive_stage(
    model,
    center_loss,
    source_x,
    source_y,
    fake_x,
    fake_y,
    validation_x,
    validation_y,
):
    x_train = np.concatenate(
        [
            source_x,
            fake_x,
        ],
        axis=0,
    )

    y_train = np.concatenate(
        [
            source_y,
            fake_y,
        ],
        axis=0,
    )

    domain = np.concatenate(
        [
            np.zeros(
                len(source_y),
                dtype=np.int64,
            ),
            np.ones(
                len(fake_y),
                dtype=np.int64,
            ),
        ]
    )

    loader = create_balanced_loader(
        x_train,
        y_train,
        domain,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.fine_tune_lr,
        weight_decay=CFG.weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=5,
        )
    )

    best_accuracy = -np.inf
    best_state = None
    patience_counter = 0

    history = []

    for epoch in range(
        1,
        CFG.finetune_epochs + 1,
    ):
        model.train()

        batch_losses = []

        for (
            x_batch,
            y_batch,
            domain_batch,
        ) in loader:

            x_batch = x_batch.to(
                DEVICE
            )

            y_batch = y_batch.to(
                DEVICE
            )

            domain_batch = domain_batch.to(
                DEVICE
            )

            optimizer.zero_grad()

            logits, features = model(
                x_batch
            )

            ce = F.cross_entropy(
                logits,
                y_batch,
            )

            center = center_loss(
                features,
                y_batch,
            )

            loss = (
                ce
                + CFG.lambda_center
                * center
            )

            if CFG.use_coral:
                source_mask = (
                    domain_batch == 0
                )

                target_mask = (
                    domain_batch == 1
                )

                if (
                    source_mask.sum() > 1
                    and target_mask.sum() > 1
                ):
                    domain_loss = coral_loss(
                        features[
                            source_mask
                        ],
                        features[
                            target_mask
                        ],
                    )

                    loss = (
                        loss
                        + CFG.lambda_coral
                        * domain_loss
                    )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            optimizer.step()

            batch_losses.append(
                float(
                    loss.detach().cpu()
                )
            )

        if (
            epoch
            % CFG.center_update_every
            == 0
        ):
            center_loss.center_shift(
                CFG.center_alpha
            )

        model.eval()

        val_predictions = []

        with torch.no_grad():
            val_loader = DataLoader(
                TensorEEGDataset(
                    validation_x
                ),
                batch_size=64,
                shuffle=False,
            )

            for batch in val_loader:
                batch = batch.to(
                    DEVICE
                )

                logits, _ = model(
                    batch
                )

                val_predictions.extend(
                    logits.argmax(
                        dim=1
                    )
                    .cpu()
                    .numpy()
                )

        val_accuracy = accuracy_score(
            validation_y,
            val_predictions,
        )

        scheduler.step(
            val_accuracy
        )

        history.append(
            {
                "epoch": epoch,
                "stage": "adaptive",
                "train_loss": float(
                    np.mean(
                        batch_losses
                    )
                ),
                "validation_accuracy": val_accuracy,
                "lr": optimizer.param_groups[
                    0
                ]["lr"],
            }
        )

        if (
            val_accuracy
            > best_accuracy
        ):
            best_accuracy = val_accuracy

            best_state = {
                key: value.detach()
                .cpu()
                .clone()
                for key, value
                in model.state_dict()
                .items()
            }

            patience_counter = 0
        else:
            patience_counter += 1

        if (
            epoch == 1
            or epoch % 5 == 0
        ):
            print(
                f"[ADAPT] "
                f"{epoch:03d}/{CFG.finetune_epochs} | "
                f"loss={np.mean(batch_losses):.4f} | "
                f"source-val={val_accuracy:.4f}"
            )

        if (
            patience_counter
            >= CFG.early_stopping_patience
        ):
            print(
                "[ADAPT] early stopping."
            )
            break

    if best_state is not None:
        model.load_state_dict(
            best_state
        )

    return (
        model,
        pd.DataFrame(history),
    )

In [8]:
# ============================================================
# CELL 9 - Full Hybrid LOSO / Target-Adaptive Experiment
# ============================================================


def choose_source_validation_subjects(
    source_subjects,
):
    n_val = max(
        1,
        int(
            round(
                len(source_subjects)
                * CFG.source_validation_subject_fraction
            )
        ),
    )

    return (
        source_subjects[-n_val:],
        source_subjects[:-n_val],
    )


def collect_subjects(
    subject_ids,
):
    xs = []
    ys = []
    subjects = []

    for subject_id in tqdm(
        subject_ids,
        desc="Loading cohort",
    ):
        x, y = cache_subject(
            subject_id
        )

        if x is None:
            continue

        xs.append(x)
        ys.append(y)

        subjects.append(
            np.full(
                len(y),
                subject_id,
                dtype=np.int64,
            )
        )

    if not xs:
        raise RuntimeError(
            "No subject data loaded."
        )

    return (
        np.concatenate(xs),
        np.concatenate(ys),
        np.concatenate(subjects),
    )


def evaluate_model(
    model,
    x,
    y,
):
    model.eval()

    predictions = []
    features = []

    loader = DataLoader(
        TensorEEGDataset(x),
        batch_size=64,
        shuffle=False,
    )

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(
                DEVICE
            )

            logits, feature = model(
                batch
            )

            predictions.append(
                logits.argmax(
                    dim=1
                )
                .cpu()
                .numpy()
            )

            features.append(
                feature.cpu().numpy()
            )

    predictions = np.concatenate(
        predictions
    )

    features = np.concatenate(
        features
    )

    return {
        "accuracy": accuracy_score(
            y,
            predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y,
            predictions,
        ),
        "predictions": predictions,
        "features": features,
    }


def run_one_target_fold(
    target_subject,
    source_subjects,
):
    print("\n" + "=" * 78)
    print(
        f"TARGET SUBJECT S{target_subject:03d}"
    )
    print("=" * 78)

    # ---------------------------
    # Target calibration/test
    # ---------------------------
    target_x_raw, target_y = cache_subject(
        target_subject
    )

    (
        target_calib_raw,
        target_calib_y,
        target_test_raw,
        target_test_y,
    ) = split_subject_calibration(
        target_x_raw,
        target_y,
        CFG.target_calibration_per_class,
        seed=SEED + target_subject,
    )

    # ---------------------------
    # Source train/validation subjects
    # ---------------------------
    source_train_subjects, source_val_subjects = (
        choose_source_validation_subjects(
            source_subjects
        )
    )

    source_train_x_raw, source_train_y, _ = (
        collect_subjects(
            source_train_subjects
        )
    )

    source_val_x_raw, source_val_y, _ = (
        collect_subjects(
            source_val_subjects
        )
    )

    print(
        "Source training subjects:",
        source_train_subjects,
    )

    print(
        "Source validation subjects:",
        source_val_subjects,
    )

    print(
        "Target calibration:",
        len(target_calib_y),
        "trials",
    )

    print(
        "Untouched target test:",
        len(target_test_y),
        "trials",
    )

    # ---------------------------
    # Source-only normalization
    # ---------------------------
    (
        source_train_x,
        norm_mean,
        norm_std,
    ) = preprocess_training_fit(
        source_train_x_raw
    )

    source_val_x = preprocess_apply(
        source_val_x_raw,
        norm_mean,
        norm_std,
    )

    target_calib_x = preprocess_apply(
        target_calib_raw,
        norm_mean,
        norm_std,
    )

    target_test_x = preprocess_apply(
        target_test_raw,
        norm_mean,
        norm_std,
    )

    # ---------------------------
    # Target FBCSP + LASSO
    # ---------------------------
    target_fbcsp = OVRFBCSPLASSO()
    target_fbcsp.fit(
        target_calib_x,
        target_calib_y,
    )

    selected_filters, selected_bands = (
        target_fbcsp.selected_filters()
    )

    print(
        "Selected sparse filters:",
        selected_filters.shape,
    )

    # ---------------------------
    # Target-specific FBGAN
    # ---------------------------
    fake_x_parts = []
    fake_y_parts = []

    if CFG.run_fbgan:
        for class_id in range(
            CFG.n_classes
        ):
            class_mask = (
                target_calib_y
                == class_id
            )

            class_x = target_calib_x[
                class_mask
            ]

            print(
                "\nFBGAN:",
                CLASS_NAMES[class_id],
                "| real calibration:",
                len(class_x),
            )

            generator = train_one_fbgan_class(
                class_x,
                selected_filters,
                selected_bands,
                epochs=CFG.gan_epochs,
                debug=(
                    CFG.debug_shapes
                    and class_id == 0
                ),
            )

            fake_class = (
                generate_fbgan_samples(
                    generator,
                    CFG.fake_per_class,
                )
            )

            fake_x_parts.append(
                fake_class
            )

            fake_y_parts.append(
                np.full(
                    len(fake_class),
                    class_id,
                    dtype=np.int64,
                )
            )

    if fake_x_parts:
        fake_x = np.concatenate(
            fake_x_parts,
            axis=0,
        )

        fake_y = np.concatenate(
            fake_y_parts,
            axis=0,
        )
    else:
        fake_x = np.empty(
            (
                0,
                CFG.channels,
                CFG.samples,
            ),
            dtype=np.float32,
        )

        fake_y = np.empty(
            (0,),
            dtype=np.int64,
        )

    print(
        "Synthetic target EEG:",
        fake_x.shape,
    )

    # ---------------------------
    # Transformer
    # ---------------------------
    model = SpatialTemporalTransformer(
        n_classes=CFG.n_classes,
        embed_dim=CFG.embed_dim,
    ).to(DEVICE)

    center_loss = (
        DiscriminativeCenterLoss(
            CFG.n_classes,
            CFG.embed_dim,
        ).to(DEVICE)
    )

    initialize_center_vectors(
        model,
        center_loss,
        source_train_x,
        source_train_y,
    )

    # ---------------------------
    # Stage 1: source pretraining
    # ---------------------------
    model, pretrain_history = (
        train_supervised_stage(
            model=model,
            center_loss=center_loss,
            train_x=source_train_x,
            train_y=source_train_y,
            validation_x=source_val_x,
            validation_y=source_val_y,
            epochs=CFG.pretrain_epochs,
            lr=CFG.classifier_lr,
            stage_name="SOURCE",
        )
    )

    # ---------------------------
    # Stage 2: adaptive fine-tuning
    # ---------------------------
    if len(fake_x) > 0:
        model, adapt_history = (
            train_adaptive_stage(
                model=model,
                center_loss=center_loss,
                source_x=source_train_x,
                source_y=source_train_y,
                fake_x=fake_x,
                fake_y=fake_y,
                validation_x=source_val_x,
                validation_y=source_val_y,
            )
        )
    else:
        adapt_history = pd.DataFrame()

    # ---------------------------
    # Final untouched target test
    # ---------------------------
    result = evaluate_model(
        model,
        target_test_x,
        target_test_y,
    )

    print(
        f"S{target_subject:03d} "
        f"accuracy={result['accuracy']:.4f} | "
        f"balanced={result['balanced_accuracy']:.4f}"
    )

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "config": asdict(CFG),
        "target_subject": target_subject,
        "source_subjects": source_subjects,
        "selected_sparse_filters": selected_filters,
        "selected_band_indices": selected_bands,
        "norm_mean": norm_mean,
        "norm_std": norm_std,
    }

    torch.save(
        checkpoint,
        RESULT_ROOT
        / f"model_target_S{target_subject:03d}.pt",
    )

    history = pd.concat(
        [
            pretrain_history,
            adapt_history,
        ],
        ignore_index=True,
    )

    return {
        "target_subject": target_subject,
        "accuracy": result["accuracy"],
        "balanced_accuracy": result[
            "balanced_accuracy"
        ],
        "target_test_y": target_test_y,
        "target_predictions": result[
            "predictions"
        ],
        "target_features": result[
            "features"
        ],
        "history": history,
        "n_sparse_filters": len(
            selected_filters
        ),
        "calibration_trials": len(
            target_calib_y
        ),
        "test_trials": len(
            target_test_y
        ),
    }


ALL_FOLD_RESULTS = []
FOLD_PREDICTIONS = []
FOLD_FEATURES = []
FOLD_HISTORIES = []

for target_subject in TARGET_SUBJECTS:
    fold_source_subjects = [
        s
        for s in SOURCE_SUBJECTS
        if s != target_subject
    ]

    fold_result = run_one_target_fold(
        target_subject,
        fold_source_subjects,
    )

    ALL_FOLD_RESULTS.append(
        fold_result
    )

    FOLD_PREDICTIONS.append(
        pd.DataFrame(
            {
                "target_subject": target_subject,
                "true": fold_result[
                    "target_test_y"
                ],
                "pred": fold_result[
                    "target_predictions"
                ],
            }
        )
    )

    FOLD_FEATURES.append(
        pd.DataFrame(
            fold_result[
                "target_features"
            ]
        ).assign(
            target_subject=target_subject,
            label=fold_result[
                "target_test_y"
            ],
        )
    )

    FOLD_HISTORIES.append(
        fold_result[
            "history"
        ].assign(
            target_subject=target_subject
        )
    )

FOLD_DF = pd.DataFrame(
    [
        {
            "target_subject": r[
                "target_subject"
            ],
            "accuracy": r[
                "accuracy"
            ],
            "balanced_accuracy": r[
                "balanced_accuracy"
            ],
            "n_sparse_filters": r[
                "n_sparse_filters"
            ],
            "calibration_trials": r[
                "calibration_trials"
            ],
            "test_trials": r[
                "test_trials"
            ],
        }
        for r in ALL_FOLD_RESULTS
    ]
)

PREDICTIONS_DF = pd.concat(
    FOLD_PREDICTIONS,
    ignore_index=True,
)

FEATURES_DF = pd.concat(
    FOLD_FEATURES,
    ignore_index=True,
)

HISTORY_DF = pd.concat(
    FOLD_HISTORIES,
    ignore_index=True,
)

SUMMARY = {
    "mean_accuracy": float(
        FOLD_DF["accuracy"].mean()
    ),
    "std_accuracy": float(
        FOLD_DF["accuracy"].std(ddof=1)
    )
    if len(FOLD_DF) > 1
    else 0.0,
    "mean_balanced_accuracy": float(
        FOLD_DF[
            "balanced_accuracy"
        ].mean()
    ),
    "n_folds": int(
        len(FOLD_DF)
    ),
    "source_subjects": SOURCE_SUBJECTS,
    "target_subjects": TARGET_SUBJECTS,
    "target_calibration_per_class": (
        CFG.target_calibration_per_class
    ),
    "fake_per_class": CFG.fake_per_class,
}

FOLD_DF.to_csv(
    RESULT_ROOT / "fold_results.csv",
    index=False,
)

PREDICTIONS_DF.to_csv(
    RESULT_ROOT / "predictions.csv",
    index=False,
)

HISTORY_DF.to_csv(
    RESULT_ROOT / "training_history.csv",
    index=False,
)

with open(
    RESULT_ROOT / "summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        SUMMARY,
        f,
        indent=2,
    )

print("\n" + "=" * 78)
print("FINAL HYBRID TRANSFORMER SUMMARY")
print("=" * 78)
print(FOLD_DF)
print(
    f"\nMean accuracy: "
    f"{SUMMARY['mean_accuracy']:.4f} "
    f"± {SUMMARY['std_accuracy']:.4f}"
)
print(
    f"Mean balanced accuracy: "
    f"{SUMMARY['mean_balanced_accuracy']:.4f}"
)


TARGET SUBJECT S103


Loading cohort:   0%|          | 0/2 [00:00<?, ?it/s]

Loading cohort:   0%|          | 0/18 [00:00<?, ?it/s]

Source training subjects: [95, 102]
Source validation subjects: [100, 88, 55, 68, 29, 31, 37, 39, 56, 63, 64, 66, 73, 24, 76, 81, 84, 93]
Target calibration: 40 trials
Untouched target test: 50 trials
Candidate dimensions: 160
Selected sparse dimensions: 64
Selected sparse filters: (64, 64)

FBGAN: Left Fist | real calibration: 10
GAN 001/30 | G=1.4611 | D=2.8006
GAN 005/30 | G=1.5361 | D=2.7406
GAN 010/30 | G=1.6834 | D=2.6405
GAN 015/30 | G=1.7825 | D=2.5799
GAN 020/30 | G=1.9714 | D=2.4506
GAN 025/30 | G=2.1727 | D=2.2741
GAN 030/30 | G=2.5667 | D=2.0150

FBGAN: Right Fist | real calibration: 10
GAN 001/30 | G=1.3575 | D=2.8011
GAN 005/30 | G=1.4614 | D=2.7139
GAN 010/30 | G=1.6389 | D=2.5880
GAN 015/30 | G=1.7195 | D=2.5610
GAN 020/30 | G=2.2549 | D=2.3491
GAN 025/30 | G=1.9924 | D=2.4124
GAN 030/30 | G=2.0952 | D=2.3475

FBGAN: Both Fists | real calibration: 10
GAN 001/30 | G=1.5130 | D=2.7183
GAN 005/30 | G=1.5374 | D=2.7058
GAN 010/30 | G=1.6898 | D=2.6080
GAN 015/30 | G=1.8702 

Loading cohort:   0%|          | 0/2 [00:00<?, ?it/s]

...

Source training subjects: [95, 102]
Source validation subjects: [100, 88, 55, 68, 29, 31, 37, 39, 56, 63, 64, 66, 73, 24, 76, 81, 84, 93]
Target calibration: 40 trials
Untouched target test: 50 trials
Candidate dimensions: 160
Selected sparse dimensions: 64
Selected sparse filters: (64, 64)

FBGAN: Left Fist | real calibration: 10
GAN 001/30 | G=1.5224 | D=2.7426
GAN 005/30 | G=1.5426 | D=2.7189
GAN 010/30 | G=1.6697 | D=2.6101
GAN 015/30 | G=1.6517 | D=2.6124
GAN 020/30 | G=1.8280 | D=2.4472
GAN 025/30 | G=2.1384 | D=2.1901
GAN 030/30 | G=2.3040 | D=1.8190

FBGAN: Right Fist | real calibration: 10
GAN 001/30 | G=1.3561 | D=2.7878
GAN 005/30 | G=1.4640 | D=2.7106
GAN 010/30 | G=1.7214 | D=2.5717
GAN 015/30 | G=1.7210 | D=2.6442
GAN 020/30 | G=1.8804 | D=2.5557
GAN 025/30 | G=1.9655 | D=2.4775
GAN 030/30 | G=1.9832 | D=2.4585

FBGAN: Both Fists | real calibration: 10
GAN 001/30 | G=1.4950 | D=2.7182
GAN 005/30 | G=1.4903 | D=2.7091
GAN 010/30 | G=1.6606 | D=2.5839
GAN 015/30 | G=1.9187 

Loading cohort:   0%|          | 0/2 [00:00<?, ?it/s]

Loading cohort:   0%|          | 0/18 [00:00<?, ?it/s]

Source training subjects: [95, 102]
Source validation subjects: [100, 88, 55, 68, 29, 31, 37, 39, 56, 63, 64, 66, 73, 24, 76, 81, 84, 93]
Target calibration: 40 trials
Untouched target test: 50 trials
Candidate dimensions: 160
Selected sparse dimensions: 64
Selected sparse filters: (64, 64)

FBGAN: Left Fist | real calibration: 10
GAN 001/30 | G=1.4083 | D=2.6876
GAN 005/30 | G=1.4577 | D=2.6491
GAN 010/30 | G=1.4767 | D=2.6528
GAN 015/30 | G=1.7691 | D=2.4912
GAN 020/30 | G=2.2013 | D=2.2898
GAN 025/30 | G=2.4841 | D=2.1013
GAN 030/30 | G=2.2111 | D=2.1807

FBGAN: Right Fist | real calibration: 10
GAN 001/30 | G=1.3933 | D=2.8155
GAN 005/30 | G=1.4863 | D=2.7295
GAN 010/30 | G=1.6462 | D=2.6340
GAN 015/30 | G=1.9903 | D=2.5104
GAN 020/30 | G=2.4703 | D=2.2995
GAN 025/30 | G=2.7214 | D=2.2579
GAN 030/30 | G=3.0322 | D=2.1162

FBGAN: Both Fists | real calibration: 10
GAN 001/30 | G=1.4134 | D=2.7853
GAN 005/30 | G=1.5144 | D=2.6930
GAN 010/30 | G=1.6973 | D=2.5703
GAN 015/30 | G=1.9827 

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 10 - Evaluation, Confusion Matrix and t-SNE
# ============================================================


def plot_results():
    plt.figure(figsize=(9, 4))

    plt.bar(
        FOLD_DF[
            "target_subject"
        ].astype(str),
        FOLD_DF[
            "accuracy"
        ],
    )

    plt.axhline(
        0.25,
        linestyle="--",
        label="4-class chance",
    )

    plt.xlabel("Held-out subject")
    plt.ylabel("Accuracy")
    plt.title(
        "Hybrid FBGAN + EEG Transformer"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))

    if "validation_accuracy" in HISTORY_DF.columns:
        for subject in HISTORY_DF[
            "target_subject"
        ].unique():
            subset = HISTORY_DF[
                HISTORY_DF[
                    "target_subject"
                ] == subject
            ]

            plt.plot(
                subset["epoch"],
                subset[
                    "validation_accuracy"
                ],
                label=f"S{subject:03d}",
            )

    plt.xlabel("Epoch")
    plt.ylabel(
        "Source validation accuracy"
    )
    plt.title(
        "Source validation curves"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_confusion():
    y_true = PREDICTIONS_DF[
        "true"
    ].to_numpy()

    y_pred = PREDICTIONS_DF[
        "pred"
    ].to_numpy()

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                CLASS_NAMES[i]
                for i in range(4)
            ],
            zero_division=0,
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
    )

    plt.figure(figsize=(7, 6))
    plt.imshow(cm)
    plt.colorbar()

    labels = [
        CLASS_NAMES[i]
        for i in range(4)
    ]

    plt.xticks(
        range(4),
        labels,
        rotation=30,
        ha="right",
    )

    plt.yticks(
        range(4),
        labels,
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(
        "Hybrid Transformer confusion matrix"
    )

    for i in range(4):
        for j in range(4):
            plt.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
            )

    plt.tight_layout()
    plt.show()


def plot_tsne():
    if len(FEATURES_DF) < 10:
        print(
            "Not enough features for t-SNE."
        )
        return

    feature_columns = [
        column
        for column in FEATURES_DF.columns
        if isinstance(column, int)
    ]

    X = FEATURES_DF[
        feature_columns
    ].to_numpy()

    y = FEATURES_DF[
        "label"
    ].to_numpy()

    perplexity = min(
        30,
        max(
            5,
            (len(X) - 1) // 4,
        ),
    )

    embedding = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=SEED,
    ).fit_transform(X)

    plt.figure(figsize=(9, 7))

    for class_id in range(
        CFG.n_classes
    ):
        mask = y == class_id

        plt.scatter(
            embedding[mask, 0],
            embedding[mask, 1],
            alpha=0.70,
            label=CLASS_NAMES[
                class_id
            ],
        )

    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.title(
        "Transformer discriminative feature space"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_results()
plot_confusion()
plot_tsne()

print("\nSaved results:")
for file in sorted(
    RESULT_ROOT.glob("*")
):
    print(file)

## Cell 11 — Recommended experimental progression

### Phase 1 — architecture smoke test

Before a long GAN run, change:

```python
CFG.n_source_subjects = 4
CFG.n_target_subjects = 1
CFG.fake_per_class = 25
CFG.gan_epochs = 5
CFG.pretrain_epochs = 5
CFG.finetune_epochs = 5
```

Verify that all shape checks and one complete fold work.

### Phase 2 — development benchmark

Restore the defaults:

```python
CFG.n_source_subjects = 20
CFG.n_target_subjects = 5
CFG.fake_per_class = 750
CFG.gan_epochs = 30
```

This has substantially more source-subject diversity than the previous four-source-subject experiment.

### Phase 3 — ablation study

Run:

```text
A  CNN-Transformer
B  CNN-Transformer + Center Loss
C  CNN-Transformer + FBGAN
D  CNN-Transformer + FBGAN + Center Loss
E  CNN-Transformer + FBGAN + Center Loss + CORAL
```

The important scientific question is whether **global temporal attention plus target-specific FBGAN adaptation** improves cross-subject generalization.

### Phase 4 — final evaluation

After hyperparameters are frozen, switch to a complete LOSO experiment over the intended evaluation cohort. Do not choose final reported test subjects because their classifier accuracy is high. Quality ranking is appropriate for a development cohort; an unbiased final subject-independent result should use a predetermined or complete evaluation cohort.

## Research basis

This notebook combines:

- **Zhang et al. (2023)** — FBGAN + FBCSP/LASSO + discriminative feature learning for subject-independent MI EEG.
- **EEGNet (Lawhern et al., 2018)** — compact EEG-specific spatial/temporal convolution.
- **EEG Conformer (Song et al., 2023)** — convolutional local feature extraction followed by Transformer self-attention for local + global EEG representation learning.

The PhysioNet EEGMMIDB contains 109 subjects with 64 EEG channels recorded at 160 Hz. MNE documents runs 4/8/12 as left-vs-right hand motor imagery and runs 6/10/14 as hands-vs-feet motor imagery.